In [ ]:
import pandas as pd

RAW = "../data/raw/"

# 1) YÜZÖLÇÜMÜ — Gazetteer (ZIP içinden okunuyor, açmana gerek yok)
gaz = pd.read_csv(RAW + "2024_Gaz_counties_national.zip",
                  sep="\t", dtype={"GEOID": str})
gaz.columns = gaz.columns.str.strip() #.strip() = baştaki ve sondaki boşlukları kırp.

# 2) NÜFUS — Census
pop = pd.read_csv(RAW + "co-est2024-alldata.csv",
                  encoding="latin-1", dtype={"STATE": str, "COUNT"
                  "Y": str})

# 3) KIRSAL/KENTSEL — CDC
nchs = pd.read_csv(RAW + "NCHSurb-rural-codes.csv", encoding="latin-1")

print("GAZ :", gaz.shape, list(gaz.columns))
print()
print("POP :", pop.shape)
print()
print("NCHS:", nchs.shape, list(nchs.columns))
print()
print("--- Gazetteer ilk 3 satır ---")
print(gaz.head(3).to_string())

GAZ : (3222, 10) ['USPS', 'GEOID', 'ANSICODE', 'NAME', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG']

POP : (3195, 83)

NCHS: (3160, 11) ['STFIPS', 'CTYFIPS', 'ST_ABBREV', 'CTYNAME', 'CBSATITLE', 'CBSAPOP', 'CTYPOP', 'CODE2023', 'CODE2013', 'CODE2006', 'CODE1990']

--- Gazetteer ilk 3 satır ---
  USPS  GEOID  ANSICODE            NAME       ALAND      AWATER  ALAND_SQMI  AWATER_SQMI   INTPTLAT  INTPTLONG
0   AL  01001    161526  Autauga County  1539631459    25677536     594.455        9.914  32.532237 -86.646440
1   AL  01003    161527  Baldwin County  4117781416  1132830835    1589.884      437.388  30.659218 -87.746067
2   AL  01005    161528  Barbour County  2292160151    50523213     885.008       19.507  31.870253 -85.405104


In [ ]:
#3 veri setini GEOID üzerinden birleştirme 
# --- GAZETTEER: GEOID zaten hazır --- 
gaz2 = gaz[["USPS", "GEOID", "NAME", "ALAND_SQMI"]]

# --- NÜFUS: sadece ilçe satırları, sonra GEOID üret ---
pop2 = pop[pop["SUMLEV"] == 50].copy()
pop2["GEOID"] = pop2["STATE"] + pop2["COUNTY"]
pop2 = pop2[["GEOID", "STNAME", "CTYNAME", "POPESTIMATE2022"]]

# --- NCHS: sayıları metne çevir, sıfır doldur, birleştir ---
nchs2 = nchs.copy()
nchs2["GEOID"] = (nchs2["STFIPS"].astype(str).str.zfill(2) #.str.zfill(2) — başına sıfır ekle/eyalet için
                  + nchs2["CTYFIPS"].astype(str).str.zfill(3))#ilçe için zfill(3)
nchs2 = nchs2[["GEOID", "ST_ABBREV", "CTYNAME", "CODE2023", "CODE2013"]]

print("GAZ :", gaz2.shape)
print("POP :", pop2.shape, "  (eyalet satırları çıkarıldı)")
print("NCHS:", nchs2.shape)
print()
print("--- Florida Hillsborough (12057) ---")
print(gaz2[gaz2.GEOID == "12057"].to_string(index=False))
print(pop2[pop2.GEOID == "12057"].to_string(index=False))
print(nchs2[nchs2.GEOID == "12057"].to_string(index=False))

GAZ : (3222, 4)
POP : (3144, 4)   (eyalet satırları çıkarıldı)
NCHS: (3160, 5)

--- Florida Hillsborough (12057) ---
USPS GEOID                NAME  ALAND_SQMI
  FL 12057 Hillsborough County    1022.537
GEOID  STNAME             CTYNAME  POPESTIMATE2022
12057 Florida Hillsborough County          1523839
GEOID ST_ABBREV             CTYNAME  CODE2023  CODE2013
12057        FL Hillsborough County       1.0       1.0


In [ ]:
# GEOID üzerinden üçlü birleştirme
ref = (gaz2
       .merge(pop2[["GEOID", "STNAME", "POPESTIMATE2022"]], on="GEOID", how="inner")#how="inner"=Sadece iki tarafta da olan satırları tut.
       .merge(nchs2[["GEOID", "CODE2023", "CODE2013"]], on="GEOID", how="left"))#how="left"=Soldaki tabloyu olduğu gibi koru. Sağdan eşleşen varsa ekle, yoksa boş bırak ama satırı silme.

print("Birleşme sonrası:", ref.shape)
print()
print("Nüfus eksik  :", ref["POPESTIMATE2022"].isna().sum())
print("NCHS eksik   :", ref["CODE2023"].isna().sum())
print()

# Nüfus yoğunluğu
ref["nufus_yogunlugu"] = (ref["POPESTIMATE2022"] / ref["ALAND_SQMI"]).round(2)

# CODE2023 boşsa CODE2013'e düş
ref["kentsel_kirsal"] = ref["CODE2023"].fillna(ref["CODE2013"])

print(ref[ref.GEOID == "12057"].to_string(index=False))

Birleşme sonrası: (3144, 8)

Nüfus eksik  : 0
NCHS eksik   : 0

USPS GEOID                NAME  ALAND_SQMI  STNAME  POPESTIMATE2022  CODE2023  CODE2013  nufus_yogunlugu  kentsel_kirsal
  FL 12057 Hillsborough County    1022.537 Florida          1523839       1.0       1.0          1490.25             1.0


In [ ]:
#Kaza versine bağlanacak anahtarı bulma 
import re

# Yönetim birimi eklerini temizle
SUFFIX = r"\s+(County|Parish|Borough|Census Area|City and Borough|Municipality|Municipio|city|City)$"

ref["county_key"] = (ref["NAME"]
                     .str.replace(SUFFIX, "", regex=True)#eki sil
                     .str.lower()#küçük harf
                     .str.replace(".", "", regex=False)#noktaları sil
                     .str.strip())#boşlukları kırp

print("Örnek dönüşümler:")
print(ref[["NAME", "county_key"]].head(5).to_string(index=False))
print()

# Aynı eyalette aynı isim var mı?
cakisma = ref[ref.duplicated(["USPS", "county_key"], keep=False)]
print("Çakışan satır sayısı:", len(cakisma))
print()
print(cakisma[["USPS", "NAME", "county_key", "POPESTIMATE2022"]]
      .sort_values(["USPS", "county_key"]).to_string(index=False))

Örnek dönüşümler:
          NAME county_key
Autauga County    autauga
Baldwin County    baldwin
Barbour County    barbour
   Bibb County       bibb
 Blount County     blount

Çakışan satır sayısı: 12

USPS             NAME county_key  POPESTIMATE2022
  MD Baltimore County  baltimore           848873
  MD   Baltimore city  baltimore           570663
  MO St. Louis County   st louis           991881
  MO   St. Louis city   st louis           286292
  VA   Fairfax County    fairfax          1140521
  VA     Fairfax city    fairfax            24825
  VA  Franklin County   franklin            55047
  VA    Franklin city   franklin             8259
  VA  Richmond County   richmond             9099
  VA    Richmond city   richmond           228670
  VA   Roanoke County    roanoke            96856
  VA     Roanoke city    roanoke            97743


In [ ]:

grup = ref.groupby(["USPS", "county_key"], as_index=False).agg(
    GEOID          = ("GEOID", "first"),
    NAME           = ("NAME", lambda s: " + ".join(s) if len(s) > 1 else s.iloc[0]),
    STNAME         = ("STNAME", "first"),
    alan_sqmi      = ("ALAND_SQMI", "sum"),
    nufus_2022     = ("POPESTIMATE2022", "sum"),
    kentsel_kirsal = ("kentsel_kirsal", "min"),
    birlesik_kayit = ("GEOID", lambda s: len(s) > 1),
)

grup["nufus_yogunlugu"] = (grup["nufus_2022"] / grup["alan_sqmi"]).round(2)

print("Önce:", len(ref), "→ Sonra:", len(grup))
print("Kalan çakışma:", grup.duplicated(["USPS", "county_key"]).sum())
print("Birleştirilen kayıt:", grup["birlesik_kayit"].sum())
print()
print(grup[grup.birlesik_kayit][["USPS", "NAME", "nufus_2022", "alan_sqmi"]].to_string(index=False))

Önce: 3144 → Sonra: 3138
Kalan çakışma: 0
Birleştirilen kayıt: 6

USPS                              NAME  nufus_2022  alan_sqmi
  MD Baltimore County + Baltimore city     1419536    679.304
  MO St. Louis County + St. Louis city     1278173    569.715
  VA     Fairfax County + Fairfax city     1165346    397.261
  VA   Franklin County + Franklin city       63306    698.888
  VA   Richmond County + Richmond city      237769    251.403
  VA     Roanoke County + Roanoke city      194599    293.069


In [7]:
CIKTI = "../data/raw/county_referans_kendi.csv"

son = grup[["USPS", "county_key", "GEOID", "NAME", "STNAME",
            "nufus_2022", "alan_sqmi", "nufus_yogunlugu",
            "kentsel_kirsal", "birlesik_kayit"]].rename(columns={"USPS": "State"})

son.to_csv(CIKTI, index=False)

print("Kaydedildi:", CIKTI)
print("Satır:", len(son), " Sütun:", son.shape[1])

Kaydedildi: ../data/raw/county_referans_kendi.csv
Satır: 3138  Sütun: 10
